Power and Intensity Tagging - Riodades
=
This notebook writes the power / current-class labels of the fixed structures of UFV Riodades. It groups the strings into continuous structure lines with the same power-intensity inside the same PST, counts their modules and writes an AutoCAD script that places one MTEXT label (e.g. `710 Wp F3 - 240 modules`) at the centre of each line.

The code is identical to the Sendim version. Only the input and output file names change (the debug script here is `teste2.scr`).

**Workflow overview:**
1. Read the string table and calculate the row (`Line`) of every string.
2. Split the strings into groups: a new group starts at a new line, a gap in the structure, a new PST or a new power-intensity.
3. Count the modules of each group (30 per string).
4. Place the label at the centre of each group and write the scripts and the Excel file.

**Input prerequisites (`Riodades estruturas.xlsx`)** (the code assumes all of these, and it doesn't check them):
- One row per string block, from an AutoCAD data extraction of the electromechanical drawing.
- Columns used: `UFV`, `PST`, `POT`, `Type`, `Position X`, `Position Y`. The other columns (`Name`, `Layer`, `INV`, `STR`, `EFX`, `GRP`, `GRP+STR`, `Type N`) are only carried along to the Excel output.
- `POT` is `<power>-<class>`, e.g. `710-3`: the first 3 characters are the power (`710 Wp`) and the last one is the current class (`F3`).
- `Type` is `T` (top string), `B` (bottom string) or `C` (centre string). T strings are inserted `1.1513` above the structure axis and B strings `1.1513` below it. Any other value is treated like `B`.
- The structures are horizontal (rows along X). The gap thresholds (40 / 30 / 20) fit the string spacing of this layout.
- Every string has 30 modules.

# Three main problems to solve:

    1 MAJOR problem:
        1. How can I group the string in lines?
            If there is a break in the structure it counts as a new line.
            If the power-intensity value changes it counts as a new line.
            If the strings are from a different PST, it counts as a new line.

    2 MINOR problems:
        1. How can I count the number of modules with their power-intensity in each line?
        2. How can I print the text in the middle of the strucutre line?

Reading the strings and calculating the line
=
- `UFV+PST` = `UFV.PST`, used to split groups at PST changes.
- `Power` = first 3 characters of `POT` + ` Wp`, `Intensity` = `F` + last character of `POT`.
- `Line` brings every string to its structure axis and rounds it to the metre:
  - `C`: `round(Y)`
  - `T`: `round(Y − 1.1513)`
  - `B` (and anything else): `round(Y + 1.1513)`

  Two strings of the same structure get the same `Line`. Because it's rounded to the metre, a structure whose axis sits close to `.5` can have its strings split into two lines.
- Sorts by `Line` descending, then `Position X` ascending (top-to-bottom, left-to-right). The grouping in the next step depends on this order.

In [1]:
import numpy as np
import pandas as pd


strings_df = pd.read_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Potência e Intensidade\Riodades estruturas.xlsx")


strings_df['UFV+PST'] = strings_df['UFV'].astype(str)+"."+strings_df['PST'].astype(str)
strings_df['Power'] = strings_df['POT'].astype(str).str[:3] + " Wp"
strings_df['Intensity'] = "F" + strings_df['POT'].astype(str).str[-1]

strings_df['Line'] = np.where(
    strings_df["Type"] == "C",
    round(strings_df["Position Y"]),
    np.where(
        strings_df["Type"] == "T",
        round(strings_df["Position Y"] - 1.1513),
        round(strings_df["Position Y"] + 1.1513)
    )
)


strings_df.sort_values(by=["Line", "Position X"], ascending=[False,True], inplace=True, ignore_index=True)
print(strings_df)

                 Name                                      Layer  UFV  PST  \
0     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
1     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
2     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
3     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
4     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
...               ...                                        ...  ...  ...   
2853  EMF309_string_2  EMF_eletromec_estrutura_fixa_705Wp_F1_1.2    1    2   
2854  EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F1_1.2    1    2   
2855  EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F1_1.2    1    2   
2856  EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F1_1.2    1    2   
2857  EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F1_1.2    1    2   

      INV  STR  EFX     GRP    GRP+STR    POT Type  Type N  Pos

Splitting the strings into groups
=
Compares each string with the previous one in the sorted table. A new group starts when any of these is true:
- `condition1`: different `Line`.
- `condition2`: X gap above `40` and neither string is `C`.
- `condition3`: X gap above `30` and exactly one of the two strings is `C`.
- `condition4`: X gap above `20` and both strings are `C`.
- `condition5`: different `UFV+PST`.
- `condition6`: different `POT` (power-intensity).

`Group` = running count of the breaks (`cumsum`), so every continuous stretch of structure with the same PST and power-intensity gets its own number. The gap thresholds are hardcoded for this layout's string spacing.

In [2]:
'''
MAJOR PROBLEM 1
'''


#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Break in the Structure with no C type on the edges
condition2 = ((abs(strings_df["Position X"] - strings_df["Position X"].shift()) > 40) &
              (strings_df["Type"] != "C") &
              (strings_df["Type"].shift() != "C"))

#Break in the Structure with one C type on the edges
condition3 = ((abs(strings_df["Position X"] - strings_df["Position X"].shift()) > 30) &
              ((strings_df["Type"] == "C") != (strings_df["Type"].shift() == "C")))

#Break in the Structure with two C types on the edges
condition4 = ((abs(strings_df["Position X"] - strings_df["Position X"].shift()) > 20) &
              (strings_df["Type"] == "C") &
              (strings_df["Type"].shift() == "C"))

#Different PSTs
condition5 = strings_df["UFV+PST"] != strings_df["UFV+PST"].shift()

#Different Power-Intensity
condition6 = strings_df["POT"] != strings_df["POT"].shift()



break_condition = condition1 | condition2 | condition3 | condition4 | condition5 | condition6

strings_df['Group'] = break_condition.cumsum()


Module count per group
=
`Module Count` = number of strings in the group × `30` (modules per string, hardcoded). Every row of the group gets the same value.

In [3]:
'''
MINOR PROBLEM 1
'''

strings_df["Module Count"] = strings_df.groupby(by="Group", sort=False)["POT"].transform('count') * 30
print(strings_df["Module Count"])

0       240
1       240
2       240
3       240
4       240
       ... 
2853    180
2854    120
2855    120
2856    120
2857    120
Name: Module Count, Length: 2858, dtype: int64


Label position and AutoCAD scripts
=
- `Text Position X` / `Text Position Y` = mean X and mean Y of the strings in the group, i.e. the centre of the structure line.
- `Script TAG` builds one MTEXT per row:
  `(command-s "._mtext" "X,Y" "h" 1.5 "j" "MC" "r" 0 "w" 0 "710 Wp F3 - 240 modules" "")`
  (height 1.5, middle-centre justification, rotation 0, no width limit). Every row of a group has the same line, so `set()` keeps one per group. The order of the lines in the file is random.
- `Script` is a debug version that writes the `Group` number instead of the label, **once per string** (not deduplicated), so the labels are stacked on top of each other.

Outputs (in the `Tagueamento Potência e Intensidade` folder):
- **Riodades - Tagueamento Potência e Intensidade.scr**: the labels. The MTEXT goes on the current layer with the current text style.
- **teste2.scr**: the group numbers, for checking the grouping.
- **Riodades - Tagueamento Potência e Intensidade.xlsx**: the full string table with `Line`, `Group`, `Module Count` and the scripts.

In [4]:
'''
MINOR PROBLEM 2
'''


strings_df["Text Position X"] = strings_df.groupby(by="Group", sort=False)["Position X"].transform('mean')
strings_df["Text Position Y"] = strings_df.groupby(by="Group", sort=False)["Position Y"].transform('mean')



strings_df["Script TAG"] = "(command-s \"._mtext\" \""+ strings_df['Text Position X'].astype(str) + "," + \
                     strings_df['Text Position Y'].astype(str) + \
                     "\" \"h\" 1.5  \"j\" \"MC\" \"r\" 0  \"w\" 0 \"" + \
                     strings_df['Power'].astype(str) + " " + strings_df['Intensity'].astype(str) + " - " + strings_df['Module Count'].astype(str) + " modules"\
                     "\" \"\")"

strings_df["Script"] = "(command-s \"._mtext\" \""+ strings_df['Text Position X'].astype(str) + "," + \
                     strings_df['Text Position Y'].astype(str) + \
                     "\" \"h\" 1.5  \"j\" \"MC\" \"r\" 0  \"w\" 0 \"" + \
                     strings_df['Group'].astype(str) +\
                     "\" \"\")"


strings_df_unique = list(set(strings_df["Script TAG"]))

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Potência e Intensidade\Riodades - Tagueamento Potência e Intensidade.scr", 'w') as f:
    f.write('\n'.join(strings_df_unique) + '\n')

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Potência e Intensidade\teste2.scr", 'w') as f:
    f.write('\n'.join(strings_df["Script"].astype(str)) + '\n')



strings_df.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Potência e Intensidade\Riodades - Tagueamento Potência e Intensidade.xlsx")
print(strings_df)

                 Name                                      Layer  UFV  PST  \
0     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
1     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
2     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
3     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
4     EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F3_3.1    3    1   
...               ...                                        ...  ...  ...   
2853  EMF309_string_2  EMF_eletromec_estrutura_fixa_705Wp_F1_1.2    1    2   
2854  EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F1_1.2    1    2   
2855  EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F1_1.2    1    2   
2856  EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F1_1.2    1    2   
2857  EMF309_string_2  EMF_eletromec_estrutura_fixa_710Wp_F1_1.2    1    2   

      INV  STR  EFX     GRP    GRP+STR    POT  ... UFV+PST   Po